In [32]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

In [33]:
CLASSES = [
    "pizza",
    "hamburger",
    "ceviche",
    "tacos",
    "steak",
    "ramen",
    "ice_cream",
    "spaghetti_bolognese",
    "fried_rice",
    "chicken_wings"
]

In [34]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([.485,.456,.406],[.229,.224,.225])
])

In [35]:
import os
from torchvision import datasets
from torch.utils.data import DataLoader, Subset

DATASET_PATH = "../data/food-101/images"

# cargar todo el dataset correctamente
full_dataset = datasets.ImageFolder(
    root=DATASET_PATH,
    transform=transform
)

# mapa de índices a clases
idx_to_class = {v: k for k, v in full_dataset.class_to_idx.items()}

# filtrar solo las clases que quieres
filtered_indices = [
    i for i, (_, label) in enumerate(full_dataset)
    if idx_to_class[label] in CLASSES
]

dataset = Subset(full_dataset, filtered_indices)

# DataLoader optimizado para GPU
loader = DataLoader(
    dataset,
    batch_size=64,      # sube batch
    shuffle=True,
    num_workers=4,      # clave
    pin_memory=True,    # clave para GPU
    persistent_workers=True  # mejora rendimiento continuo
)

print("Total imágenes:", len(dataset))
print("Clases usadas:", CLASSES)

Total imágenes: 10000
Clases usadas: ['pizza', 'hamburger', 'ceviche', 'tacos', 'steak', 'ramen', 'ice_cream', 'spaghetti_bolognese', 'fried_rice', 'chicken_wings']


In [36]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)

# congelar TODO
for param in model.parameters():
    param.requires_grad = False

# reemplazar última capa
model.classifier[1] = torch.nn.Linear(1280, len(CLASSES))

# entrenar SOLO la última capa
for param in model.classifier.parameters():
    param.requires_grad = True

In [37]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
# congelar modelo base
for param in model.parameters():
    param.requires_grad = False

# solo entrenar la última capa
for param in model.classifier.parameters():
    param.requires_grad = True

optimizer = torch.optim.Adam(model.classifier.parameters(), lr=0.001)

EPOCHS = 20

# entrenamiento optimizado
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss:.4f}")

Epoch 1/20, Loss: 208.2261
Epoch 2/20, Loss: 134.2919
Epoch 3/20, Loss: 119.4704
Epoch 4/20, Loss: 110.1493
Epoch 5/20, Loss: 104.6693
Epoch 6/20, Loss: 102.5179
Epoch 7/20, Loss: 100.8241
Epoch 8/20, Loss: 98.9600
Epoch 9/20, Loss: 97.4114
Epoch 10/20, Loss: 94.9358
Epoch 11/20, Loss: 92.1317
Epoch 12/20, Loss: 92.2508
Epoch 13/20, Loss: 91.9416
Epoch 14/20, Loss: 90.3203
Epoch 15/20, Loss: 88.8424
Epoch 16/20, Loss: 89.9650
Epoch 17/20, Loss: 88.8423
Epoch 18/20, Loss: 87.5561
Epoch 19/20, Loss: 86.7347
Epoch 20/20, Loss: 87.1753


In [38]:
torch.save(model.state_dict(), "../models/modelo_efficientnet.pth")
print("Modelo guardado correctamente")

Modelo guardado correctamente
